# EduMentor AI - Data Cleaning & Preprocessing

This notebook prepares a clean student-level dataset from OULAD for risk prediction.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

table_files = {
    'student_info': 'studentInfo.csv',
    'student_registration': 'studentRegistration.csv',
    'student_assessment': 'studentAssessment.csv',
    'assessments': 'assessments.csv',
    'student_vle': 'studentVle.csv',
    'courses': 'courses.csv',
}

tables = {}
for name, filename in table_files.items():
    fp = DATA_RAW / filename
    if fp.exists():
        tables[name] = pd.read_csv(fp)

if 'student_info' not in tables:
    raise FileNotFoundError('studentInfo.csv is required in data/raw')

{k: v.shape for k, v in tables.items()}

In [ ]:
# Build student-level feature table
student_info = tables['student_info'].copy()
key_cols = ['code_module', 'code_presentation', 'id_student']
df = student_info.copy()

if 'student_registration' in tables:
    registration = tables['student_registration'].copy()
    reg_agg = registration.groupby(key_cols, dropna=False).agg(
        registration_day=('date_registration', 'mean'),
        unregister_day=('date_unregistration', 'mean'),
    ).reset_index()
    df = df.merge(reg_agg, on=key_cols, how='left')

if {'student_assessment', 'assessments'}.issubset(set(tables.keys())):
    assessment = tables['student_assessment'].copy()
    assessments = tables['assessments'].copy()
    assessment = assessment.merge(
        assessments[['id_assessment', 'assessment_type', 'weight']],
        on='id_assessment',
        how='left',
    )
    ass_agg = assessment.groupby(key_cols, dropna=False).agg(
        avg_assessment_score=('score', 'mean'),
        submitted_assessments=('is_banked', 'count'),
        avg_assessment_weight=('weight', 'mean'),
    ).reset_index()
    df = df.merge(ass_agg, on=key_cols, how='left')

if 'student_vle' in tables:
    student_vle = tables['student_vle'].copy()
    vle_agg = student_vle.groupby(key_cols, dropna=False).agg(
        total_clicks=('sum_click', 'sum'),
        avg_clicks_per_event=('sum_click', 'mean'),
        activity_events=('sum_click', 'count'),
    ).reset_index()
    df = df.merge(vle_agg, on=key_cols, how='left')

if 'final_result' not in df.columns:
    raise ValueError('final_result column not found in studentInfo.csv')

# Risk target: 1 = At Risk (Fail/Withdrawn), 0 = Safe
df['risk_label'] = df['final_result'].isin(['Fail', 'Withdrawn']).astype(int)
df.head()

In [ ]:
# Basic EDA checkpoints
missing_summary = df.isna().mean().sort_values(ascending=False).head(20)
class_distribution = df['risk_label'].value_counts(normalize=True)
print('Top missingness:')
print(missing_summary)
print('\nClass distribution (risk_label):')
print(class_distribution)

In [ ]:
# Save cleaned dataset and train/test split for modeling
clean_path = DATA_PROCESSED / 'clean_data.csv'
df.to_csv(clean_path, index=False)

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['risk_label']
)

train_df.to_csv(DATA_PROCESSED / 'train_data.csv', index=False)
test_df.to_csv(DATA_PROCESSED / 'test_data.csv', index=False)

print(f'Saved: {clean_path}')
print(f'Train rows: {len(train_df)} | Test rows: {len(test_df)}')